# 实战案例 1：倒立摆平衡控制（CartPole）

**真实场景**：物流 AGV 车体平衡、教学机器人、一级倒立摆原型机——状态为小车位置/速度、杆角度/角速度，动作为左右推力。

**本 Notebook 目标**：
1. 用 Stable-Baselines3 PPO 训练 `CartPole-v1`
2. 绘制学习曲线并评估策略
3. 理解「仿真验证 → 再迁移到真实硬件」的第一步

**依赖**：`pip install gymnasium stable-baselines3 matplotlib numpy`

In [ ]:
!pip install -q gymnasium stable-baselines3 matplotlib numpy

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.evaluation import evaluate_policy

SEED = 42
TOTAL_STEPS = 50_000  # 正式实验可改为 200_000

## 1. 环境冒烟测试

In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=SEED)
print("obs shape:", obs.shape, "action space:", env.action_space)
for _ in range(5):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
env.close()

## 2. 训练 PPO 并记录回报

In [ ]:
class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.ep_returns = []
        self._ep = 0
    def _on_step(self):
        infos = self.locals.get("infos", [])
        for info in infos:
            if "episode" in info:
                self.ep_returns.append(info["episode"]["r"])
        return True

train_env = gym.make("CartPole-v1")
logger = RewardLogger()
model = PPO("MlpPolicy", train_env, verbose=0, seed=SEED)
model.learn(total_timesteps=TOTAL_STEPS, callback=logger)
train_env.close()
print(f"完成训练，共 {len(logger.ep_returns)} 个 episode")

In [ ]:
def moving_average(x, w=50):
    if len(x) < w:
        return np.array(x)
    return np.convolve(x, np.ones(w)/w, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(moving_average(logger.ep_returns, 30), label="MA-30 return")
plt.axhline(475, color="r", ls="--", label="solved threshold (~475)")
plt.xlabel("Episode"); plt.ylabel("Return"); plt.legend(); plt.title("CartPole PPO Learning Curve")
plt.tight_layout(); plt.show()

## 3. 确定性评估（上线前必做）

In [ ]:
eval_env = gym.make("CartPole-v1")
mean_ret, std_ret = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
eval_env.close()
print(f"Eval: mean={mean_ret:.1f} ± {std_ret:.1f} (max=500)")
assert mean_ret > 400, "未收敛，请增加 TOTAL_STEPS 或检查 seed"

## 4. 与随机策略对比

In [ ]:
random_env = gym.make("CartPole-v1")
rand_rets = []
for ep in range(20):
    obs, _ = random_env.reset(seed=SEED + ep)
    done, ret = False, 0
    while not done:
        obs, r, term, trunc, _ = random_env.step(random_env.action_space.sample())
        ret += r; done = term or trunc
    rand_rets.append(ret)
random_env.close()
print(f"Random policy mean return: {np.mean(rand_rets):.1f}")
print(f"PPO improvement: {mean_ret - np.mean(rand_rets):.1f}")

## 5. 落地思考（边跑边想）

| 难点 | 本案例中的对应 | 下一步真实部署 |
|------|----------------|----------------|
| 仿真与真机差异 | Gym 理想动力学 | 加摩擦/延迟/噪声 Wrapper |
| 安全 | 无碰撞约束 | 限幅、急停、护栏策略 |
| 样本效率 | 5 万步即可 | 真机需 Sim2Real 或 IL 预训练 |

保存模型：`model.save("ppo_cartpole_case1")`

In [ ]:
model.save("ppo_cartpole_case1")
print("模型已保存")